# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR\(^2\) dataset using the `mlcroissant` library.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and sample records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show metadata: dataset name and description
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.date_published}")

## 2. Data Overview
List available record sets in the dataset, and inspect their fields and columns by their `@id`.

In [ ]:
# Discover record sets
record_sets = dataset.metadata.record_sets
print(f"Record sets in the dataset:")
for rec in record_sets:
    print(f"  @id: {rec.id}\n    name: {rec.name}\n    description: {rec.description}")

# Print fields for each record set
print("\nRecord set fields (by @id):")
for rec in record_sets:
    print(f"\nRecord set @id: {rec.id}")
    for field in rec.fields:
        print(f"  field @id: {field.id} | name: {field.name} | dataType: {field.data_type}")

## 3. Data Extraction
Load tabular record set(s) into pandas DataFrame(s) for exploration. Use `@id` for record sets and fields.

In [ ]:
# Extract all record sets into DataFrames by their @id
dataframes = {}
for rec in record_sets:
    rec_id = rec.id
    print(f"Loading records for record set '{rec_id}' ...")
    records = list(dataset.records(record_set=rec_id))
    # Only load if there is data
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rec_id] = df
        print(f"Loaded {len(df)} records and columns: {df.columns.tolist()}")
    else:
        print("(no records found or empty record set)")

# Show columns of the first record set with data
main_rec_id = None
for rec_id, df in dataframes.items():
    if df.shape[1] > 0:
        main_rec_id = rec_id
        print(f"\nFields (@id) in DataFrame for record set '{main_rec_id}':\n{df.columns.tolist()}")
        display(df.head())
        break

# Save the list of loaded record set @ids
record_set_ids = list(dataframes.keys())

## 4. Exploratory Data Analysis (EDA)
Apply basic processing: filter records on a numeric field, normalize, group, etc., referencing by `@id`.

In [ ]:
# Choose a numeric field for EDA
# We'll look for a float/int typed field in the first available table
main_rec = None
for rec in record_sets:
    if rec.id == main_rec_id:
        main_rec = rec
        break

numeric_field_id = None
group_field_id = None
for field in main_rec.fields:
    if field.data_type in ('Float', 'Integer', 'Number') and not numeric_field_id:
        numeric_field_id = field.id
    # For grouping, try to find a Text/categorical field
    if field.data_type == 'Text' and not group_field_id:
        group_field_id = field.id

if numeric_field_id is None:
    print("No numeric field found for EDA in main record set. Dataset exploration ends here.")
else:
    print(f"Using numeric field @id: {numeric_field_id}")
    df = dataframes[main_rec_id]

    # Convert numeric in case it's loaded as string
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Basic filtering: e.g., show all records where value > 10
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}: (showing 5)")
    display(filtered_df.head())

    # Normalize the selected numeric field
    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, normalized_col]].head())

    # Optional: group by a categorical field if present
    if group_field_id and group_field_id in df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        display(grouped.head())

## 5. Visualization
Basic visualizations of numeric distributions and groupings.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and main_rec_id in dataframes:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("count")
    plt.show()

    # If group_field_id exists, boxplot grouped values
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook has loaded the FAIR\(^2\) mass spectrometry dataset using the Croissant schema. We have reviewed available record sets, explored and filtered data by record set and field `@id`, normalized a numeric attribute, and visualized distributions. You can use the loaded DataFrames and field IDs for further in-depth analysis relevant to your research questions.
